# Experimento 06 — Conversão e validação Keras → TensorFlow Lite FP16

Objetivos:
1. carregar `melhor_modelo_exp06.keras`;
2. converter para TFLite com pesos FP16;
3. comparar tamanhos FP32 × FP16;
4. validar numericamente Keras FP32 × TFLite FP16;
5. comparar TFLite FP32 × FP16 quando a saída anterior estiver disponível;
6. gerar os artefatos de referência para a Raspberry Pi Zero 2 W.


In [1]:
import os
import numpy as np
import tensorflow as tf

print("TensorFlow:", tf.__version__)
print("NumPy:", np.__version__)


TensorFlow: 2.21.0
NumPy: 2.5.1


In [2]:
KERAS_MODEL_PATH = "melhor_modelo_exp06.keras"
TFLITE_FP32_PATH = "melhor_modelo_exp06.tflite"
TFLITE_FP16_PATH = "melhor_modelo_exp06_fp16.tflite"

BENCHMARK_INPUT_PATH = "benchmark_input.npy"
KERAS_OUTPUT_PATH = "benchmark_output_keras.npy"
TFLITE_FP32_OUTPUT_PATH = "benchmark_output_tflite.npy"
TFLITE_FP16_OUTPUT_PATH = "benchmark_output_tflite_fp16.npy"

assert os.path.exists(KERAS_MODEL_PATH), f"Arquivo não encontrado: {KERAS_MODEL_PATH}"


## Carregamento do modelo Keras


In [3]:
model = tf.keras.models.load_model(KERAS_MODEL_PATH, compile=False)
model.summary()


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, None,      │          0 │ -                 │
│ (InputLayer)        │ None, 1)          │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d (Conv2D)     │ (None, None,      │        640 │ input_layer[0][0] │
│                     │ None, 64)         │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_1 (Conv2D)   │ (None, None,      │     36,928 │ conv2d[0][0]      │
│                     │ None, 64)         │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_2 (Conv2D)   │ (None, None,      │     36,928 │ conv2d_1[0][0]    │
│                     │ None, 64)         │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_3 (Conv2D)   │ (None, None,      │     36,928 │ conv2d_2[0][0]    │
│                     │ None, 64)         │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_4 (Conv2D)   │ (None, None,      │     36,928 │ conv2d_3[0][0]    │
│                     │ None, 64)         │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_5 (Conv2D)   │ (None, None,      │     36,928 │ conv2d_4[0][0]    │
│                     │ None, 64)         │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ predicted_noise     │ (None, None,      │        577 │ conv2d_5[0][0]    │
│ (Conv2D)            │ None, 1)          │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ restored_image      │ (None, None,      │          0 │ input_layer[0][0… │
│ (Subtract)          │ None, 1)          │            │ predicted_noise[… │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 185,857 (726.00 KB)

 Trainable params: 185,857 (726.00 KB)

 Non-trainable params: 0 (0.00 B)

## Conversão FP16

A otimização mantém a interface de entrada/saída em `float32`, mas reduz para `float16` os pesos elegíveis armazenados no arquivo TFLite.


In [4]:
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.target_spec.supported_types = [tf.float16]

tflite_fp16_model = converter.convert()

with open(TFLITE_FP16_PATH, "wb") as f:
    f.write(tflite_fp16_model)

print("Modelo salvo:", TFLITE_FP16_PATH)


INFO:tensorflow:Assets written to: C:\Users\cayoc\AppData\Local\Temp\tmpmsqcnrhf\assets


INFO:tensorflow:Assets written to: C:\Users\cayoc\AppData\Local\Temp\tmpmsqcnrhf\assets


Saved artifact at 'C:\Users\cayoc\AppData\Local\Temp\tmpmsqcnrhf'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, None, None, 1), dtype=tf.float32, name='input_layer')
Output Type:
  TensorSpec(shape=(None, None, None, 1), dtype=tf.float32, name=None)
Captures:
  2598244335888: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2598244337232: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2598244336656: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2598244337808: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2598244337616: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2598244336848: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2598244338000: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2598244338576: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2598244338384: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2598244338960: TensorSpec(shape=(), dtype=tf.resource, name=

## Comparação de tamanho


In [5]:
def kib(path):
    return os.path.getsize(path) / 1024

print(f"Keras:       {kib(KERAS_MODEL_PATH):.2f} KiB")
print(f"TFLite FP16: {kib(TFLITE_FP16_PATH):.2f} KiB")

if os.path.exists(TFLITE_FP32_PATH):
    fp32 = kib(TFLITE_FP32_PATH)
    fp16 = kib(TFLITE_FP16_PATH)
    print(f"TFLite FP32: {fp32:.2f} KiB")
    print(f"Redução:     {100*(1-fp16/fp32):.2f}%")


Keras:       2229.80 KiB
TFLite FP16: 370.51 KiB
TFLite FP32: 730.94 KiB
Redução:     49.31%


## Reutilização da mesma entrada de benchmark


In [6]:
assert os.path.exists(BENCHMARK_INPUT_PATH), (
    "benchmark_input.npy não encontrado. "
    "Use o mesmo arquivo da validação FP32 para preservar a comparabilidade."
)

entrada = np.load(BENCHMARK_INPUT_PATH).astype(np.float32)
print("Entrada:", entrada.shape, entrada.dtype)


Entrada: (1, 128, 128, 1) float32


## Saída Keras FP32 de referência


In [7]:
saida_keras = model(entrada, training=False).numpy().astype(np.float32)
np.save(KERAS_OUTPUT_PATH, saida_keras)
print("Saída Keras:", saida_keras.shape, saida_keras.dtype)


Saída Keras: (1, 128, 128, 1) float32


## Inferência TFLite FP16 no PC


In [8]:
interpreter = tf.lite.Interpreter(model_path=TFLITE_FP16_PATH)
inp = interpreter.get_input_details()

interpreter.resize_tensor_input(inp[0]["index"], entrada.shape, strict=False)
interpreter.allocate_tensors()

inp = interpreter.get_input_details()
out = interpreter.get_output_details()

print("Entrada:", inp)
print("\nSaída:", out)

interpreter.set_tensor(inp[0]["index"], entrada)
interpreter.invoke()
saida_fp16 = interpreter.get_tensor(out[0]["index"]).astype(np.float32)

np.save(TFLITE_FP16_OUTPUT_PATH, saida_fp16)
print("\nSaída FP16 salva:", TFLITE_FP16_OUTPUT_PATH)


Entrada: [{'name': 'serving_default_input_layer:0', 'index': 0, 'shape': array([  1, 128, 128,   1], dtype=int32), 'shape_signature': array([-1, -1, -1,  1], dtype=int32), 'dtype': <class 'numpy.float32'>, 'quantization': (0.0, 0), 'quantization_parameters': {'scales': array([], dtype=float32), 'zero_points': array([], dtype=int32), 'quantized_dimension': 0, 'block_size': 0}, 'sparsity_parameters': {}}]

Saída: [{'name': 'StatefulPartitionedCall_1:0', 'index': 36, 'shape': array([  1, 128, 128,   1], dtype=int32), 'shape_signature': array([-1, -1, -1,  1], dtype=int32), 'dtype': <class 'numpy.float32'>, 'quantization': (0.0, 0), 'quantization_parameters': {'scales': array([], dtype=float32), 'zero_points': array([], dtype=int32), 'quantized_dimension': 0, 'block_size': 0}, 'sparsity_parameters': {}}]

Saída FP16 salva: benchmark_output_tflite_fp16.npy


c:\Doutorado\termografia\.venv\Lib\site-packages\tensorflow\lite\python\interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


## Equivalência numérica

Diferenças em relação ao Keras FP32 são esperadas, pois os pesos elegíveis foram armazenados em menor precisão.


In [9]:
def comparar(ref, teste, titulo):
    d = teste.astype(np.float64) - ref.astype(np.float64)
    a = np.abs(d)
    print(f"--- {titulo} ---")
    print(f"Máxima diferença absoluta: {np.max(a):.10e}")
    print(f"Média diferença absoluta:  {np.mean(a):.10e}")
    print(f"RMSE entre as saídas:      {np.sqrt(np.mean(d**2)):.10e}")
    for tol in (1e-5, 1e-4, 1e-3):
        print(f"allclose ({tol:.0e}):", np.allclose(teste, ref, rtol=tol, atol=tol))

comparar(saida_keras, saida_fp16, "Keras FP32 × TFLite FP16")


--- Keras FP32 × TFLite FP16 ---
Máxima diferença absoluta: 1.0204315186e-04
Média diferença absoluta:  1.4164772203e-05
RMSE entre as saídas:      1.8333404105e-05
allclose (1e-05): False
allclose (1e-04): True
allclose (1e-03): True


## Comparação TFLite FP32 × TFLite FP16


In [10]:
if os.path.exists(TFLITE_FP32_OUTPUT_PATH):
    saida_fp32 = np.load(TFLITE_FP32_OUTPUT_PATH).astype(np.float32)
    comparar(saida_fp32, saida_fp16, "TFLite FP32 × TFLite FP16")
else:
    print("benchmark_output_tflite.npy não encontrado; comparação ignorada.")


--- TFLite FP32 × TFLite FP16 ---
Máxima diferença absoluta: 1.0216236115e-04
Média diferença absoluta:  1.4165337838e-05
RMSE entre as saídas:      1.8334477476e-05
allclose (1e-05): False
allclose (1e-04): True
allclose (1e-03): True


## Artefatos para a Raspberry Pi Zero 2 W

Envie:
- `melhor_modelo_exp06_fp16.tflite`
- `benchmark_input.npy`
- `benchmark_output_tflite_fp16.npy`

Primeiro validaremos PC × Raspberry com o mesmo modelo FP16. Depois repetiremos o protocolo de 10 warm-ups + 100 inferências para comparar diretamente com a baseline FP32.
